Airbnb New york Exploratory Data Analysis

***1. Importing Libraries***

```python

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

***2. Loading the Dataset***

```python

In [ ]:
data = pd.read_csv(
    'new_york_listings_2024.csv',
    encoding_errors='ignore',
  
)

***3. Initial Exploration of the dataset***

```python

In [ ]:
data.head(5)

In [ ]:
data.shape

In [ ]:
data.info()

In [ ]:
#statitstical Summary of the dataset
data.describe()

***4. Data Cleaning***

```python

In [ ]:

# Dropping rows with any missing values
data.dropna(inplace=True)

data.isnull().sum()



In [ ]:
data.drop_duplicates(inplace=True)
data.duplicated().sum()

In [ ]:
data.shape




In [ ]:
data.dtypes
data['id']=data['id'].astype(object)
data['host_id']=data['host_id'].astype(object)
data.dtypes

In [ ]:
# Replace 'Studio' in the 'bedrooms' column with the mean number of bedrooms (excluding 'Studio')
mean_bedrooms = pd.to_numeric(data.loc[data['bedrooms'] != 'Studio', 'bedrooms'], errors='coerce').mean()
data['bedrooms'] = data['bedrooms'].replace('Studio', round(mean_bedrooms, 1))
data['bedrooms'] = data['bedrooms'].astype('int64')
data['bedrooms'].unique()

In [ ]:
data.drop(columns=['last_review'],inplace=True)

In [ ]:
data['price'].unique()


In [ ]:

data['host_name'].unique()


In [ ]:

data['neighbourhood_group'].unique()


In [ ]:
data['neighbourhood'].unique()


In [ ]:
data['room_type'].unique()


In [ ]:
data['beds'].unique()

In [ ]:
data['baths'].unique()

In [ ]:
mean_baths = pd.to_numeric(data.loc[data['baths'] != 'Not specified', 'baths'], errors='coerce').mean()
data['baths'] = data['baths'].replace('Not specified', round(mean_baths, 1))
data['baths'] = pd.to_numeric(data['baths'], errors='coerce').astype('int64')

In [ ]:
data['rating'].unique()
mean_rating = pd.to_numeric(
	data.loc[~data['rating'].isin(['New ', 'No rating']), 'rating'],
	errors='coerce'
).mean()
data['rating'] = data['rating'].replace('New ', round(mean_rating, 2))
data['rating'] = data['rating'].replace('No rating', round(mean_rating, 2))
data['rating'] = pd.to_numeric(data['rating'], errors='coerce').astype('float64')
data['rating'].unique()

In [ ]:
data['license'].unique()
def simplify_license(value):
    if pd.isna(value):
        return 'Not Licensed'
    elif str(value).strip().lower() in ['no license', 'exempt']:
        return 'Not Licensed'
    else:
        return 'Licensed'

data['license_status'] = data['license'].apply(simplify_license)
data['license_status'].unique()

EDA


***5. Univariate Analysis***

```python

In [ ]:
data['price']

In [ ]:
sns.boxplot(data=data,x='price')

In [ ]:
df=data[data['price']<1500]
sns.boxplot(data=df,x='price')

In [ ]:
plt.figure(figsize=(10,6))
sns.histplot(data=df,x='price',bins=100, kde=True)
plt.ylabel('Frequency')
plt.xlabel('Price')
plt.title('Price Distribution')

In [ ]:
plt.figure(figsize=(10,6))
sns.histplot(data=df,x='availability_365', kde=True)
plt.ylabel('Frequency')
plt.xlabel('Availability in 365 days')

In [ ]:
plt.figure(figsize=(10,6))
sns.histplot(df['minimum_nights'],bins=50)
plt.xlim(0, 500)  # zoom in on common values
plt.xlabel('Minimum Nights')
plt.ylabel('Frequency')
plt.title('Minimum Nights Distribution')

In [ ]:
df=df[df['calculated_host_listings_count']<300]
sns.boxplot(x=df['calculated_host_listings_count'])
plt.figure(figsize=(8,8))


In [ ]:
plt.figure(figsize=(10,6))
sns.countplot(x='beds', data=df)
plt.xlabel('Number of Beds')
plt.ylabel('Count')
plt.title('Distribution of Beds')

In [ ]:
plt.figure(figsize=(8,8))
room_type_counts = data['room_type'].value_counts()
plt.pie(room_type_counts, labels=room_type_counts.index, autopct='%1.1f%%', startangle=140)
plt.title('Distribution of Room Types')
plt.show()

In [ ]:
top_neigh = df['neighbourhood'].value_counts().nlargest(10).index
plt.figure(figsize=(12, 6))
plt.pie(df['neighbourhood'].value_counts().loc[top_neigh], labels=top_neigh, autopct='%1.1f%%', startangle=140)
plt.title('Top 10 Neighbourhoods by Percentage')


In [ ]:
df.groupby(by='neighbourhood_group')['price'].mean().sort_values(ascending=False).plot(kind='bar', figsize=(10, 6))
plt.title('Average Price by Neighbourhood Group')

In [ ]:
df['license_status'].value_counts().plot.pie(autopct='%1.1f%%', startangle=90, colors=['#66c2a5','#fc8d62'])
plt.title("Proportion of Licensed Listings")
plt.ylabel("")  # Hide Y-axis label
plt.show()


***6.Feature Engineering***

In [ ]:
df['price_per_bed'] = df['price'] / df['beds']
df.head()

In [ ]:
df.groupby(by='neighbourhood_group')['price_per_bed'].mean()

***Bivariate Analysis***
one variable dependency on another variable

```python

In [ ]:
sns.barplot(x='neighbourhood_group', y='price', data=df,hue='room_type')
plt.figure(figsize=(10, 6))

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df,x='number_of_reviews', y='price', hue='neighbourhood_group')
plt.title('Price vs Number of Reviews by Neighbourhood Group')
plt.xlabel('Number of Reviews')
plt.ylabel('Price')

In [ ]:
plt.figure(figsize=(10, 8))
sns.pairplot(data=df, vars=['price', 'minimum_nights', 'number_of_reviews', 'availability_365'], hue='room_type')

In [ ]:
plt.figure(figsize=(10, 8))
sns.scatterplot(x='longitude', y='latitude', data=df, hue='room_type')

plt.title('Geographical Distribution of Room Types')
plt.figure(figsize=(10, 8))
sns.scatterplot(x='longitude', y='latitude', data=df, hue='neighbourhood_group')
plt.title('Geographical Distribution of Neighbourhood Groups')

***7.map plotting with folium***

```python

In [ ]:
pip install folium

In [ ]:
#import folium
import folium
from folium.plugins import MarkerCluster




In [ ]:
# Center the map on the average coordinates
center_lat = df['latitude'].mean()
center_lon = df['longitude'].mean()


In [ ]:

colors = {
    'Manhattan': 'red',
    'Brooklyn': 'blue',
    'Queens': 'green',
    'Bronx': 'purple',
    'Staten Island': 'orange'
}

In [ ]:
airbnb_map = folium.Map(location=[center_lat, center_lon], zoom_start=11, tiles='CartoDB positron')
# Clustered markers

marker_cluster = MarkerCluster().add_to(airbnb_map)

# Add markers to map
for _, row in df.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=3,
        color=colors.get(row['neighbourhood_group'], 'gray'),
        fill=True,
        fill_color=colors.get(row['neighbourhood_group'], 'gray'),
        fill_opacity=0.6,
        popup=f"{row['neighbourhood']}<br>Price: ${row['price']}<br>Room: {row['room_type']}"
    ).add_to(marker_cluster)


airbnb_map

In [ ]:
airbnb_map.save("airbnb_map.html")

***8.Statistical Analysis***

```python

In [ ]:
df.describe(include='number')


Exporting The Cleaned Dataset

```python


In [ ]:
df.to_csv('airbnb_cleaned.csv', index=False)
